In [1]:
import os
base = "/kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2"
print("ملفات الجذر:", sorted(f for f in os.listdir(base) if os.path.isfile(f"{base}/{f}")))
for root, dirs, files in os.walk(f"{base}/sample9000"):
    if root[len(base):].count("/") <= 3:
        print(root[len(base):], "|", len(dirs), "dirs,", len(files), "files")

ملفات الجذر: ['__notebook__.ipynb', '__output__.json', '__results__.html', 'custom.css', 'data.yaml', 'sample9000_list.txt']
/sample9000 | 2 dirs, 0 files
/sample9000/val | 2 dirs, 0 files
/sample9000/val/labels | 0 dirs, 1977 files
/sample9000/val/images | 0 dirs, 1977 files
/sample9000/train | 2 dirs, 0 files
/sample9000/train/labels | 0 dirs, 8999 files
/sample9000/train/images | 0 dirs, 8999 files


In [2]:
base = "/kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2"
print(open(f"{base}/data.yaml").read())

import os
from collections import Counter
names = os.listdir(f"{base}/sample9000/train/images")
print(Counter(n.split("_")[0] for n in names))

path: /kaggle/working/sample9000
train: train/images
val: val/images
names:
  0: D00
  1: D10
  2: D20
  3: D40

Counter({'Japan': 4000, 'India': 3500, 'Czech': 1499})


In [3]:
import os, numpy as np
from collections import Counter

ROOT = "/kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2/sample9000"
names = ["D00", "D10", "D20", "D40"]

def build_vectors(split):
    lbl_dir = f"{ROOT}/{split}/labels"
    stems, vecs = [], []
    for f in sorted(os.listdir(lbl_dir)):
        v = np.zeros(4, dtype=np.float32)
        for line in open(f"{lbl_dir}/{f}").read().strip().splitlines():
            if line.strip():
                v[int(line.split()[0])] = 1
        stems.append(f[:-4]); vecs.append(v)
    return stems, np.stack(vecs)

train_stems, train_y = build_vectors("train")
val_stems, val_y = build_vectors("val")

print(train_y.shape, val_y.shape)
for split, y in [("train", train_y), ("val", val_y)]:
    empty = int((y.sum(axis=1) == 0).sum())
    print(split, "| صور:", len(y), "| فاضية:", empty)
    for i, n in enumerate(names):
        print(f"   {n}: {int(y[:, i].sum())} صورة ({y[:, i].mean()*100:.0f}%)")

(8999, 4) (1977, 4)
train | صور: 8999 | فاضية: 1427
   D00: 2765 صورة (31%)
   D10: 2600 صورة (29%)
   D20: 3456 صورة (38%)
   D40: 3074 صورة (34%)
val | صور: 1977 | فاضية: 43
   D00: 1419 صورة (72%)
   D10: 738 صورة (37%)
   D20: 486 صورة (25%)
   D40: 164 صورة (8%)


In [4]:

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class RoadDataset(Dataset):
    def __init__(self, split, stems, y):
        self.split, self.stems, self.y = split, stems, y
    def __len__(self):
        return len(self.stems)
    def __getitem__(self, i):
        img = Image.open(f"{ROOT}/{self.split}/images/{self.stems[i]}.jpg").convert("RGB")
        return tf(img), torch.tensor(self.y[i])

train_ds = RoadDataset("train", train_stems, train_y)
val_ds = RoadDataset("val", val_stems, val_y)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

x, y = next(iter(train_dl))
print(x.shape, y.shape)

torch.Size([32, 3, 384, 384]) torch.Size([32, 4])


In [5]:
import torch
print(torch.cuda.is_available())

True


In [6]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda")
torch.manual_seed(42)

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 4)
    for p in m.parameters():
        p.requires_grad = False
    for p in m.fc.parameters():
        p.requires_grad = True
    return m.to(device)

model = build_model()
print("trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad),
      "| total:", sum(p.numel() for p in model.parameters()))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s] 


trainable: 2052 | total: 11178564


In [7]:
import copy, numpy as np, torch.optim as optim
from sklearn.metrics import roc_auc_score

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

def run_epoch(dl, train=True):
    model.train(train)
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * x.size(0); n += x.size(0)
    return total / n

@torch.no_grad()
def predict(dl):
    model.eval()
    return np.concatenate([torch.sigmoid(model(x.to(device))).cpu().numpy() for x, _ in dl])

best_va, best_state = 1e9, None
for epoch in range(1, 9):
    tr = run_epoch(train_dl, True)
    va = run_epoch(val_dl, False)
    print(f"epoch {epoch} | train {tr:.4f} | val {va:.4f}")
    if va < best_va:
        best_va, best_state = va, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
val_probs = predict(val_dl)
print("best val loss:", round(best_va, 4))
for i, n in enumerate(["D00", "D10", "D20", "D40"]):
    print(n, "val AUC:", round(roc_auc_score(val_y[:, i], val_probs[:, i]), 3))

epoch 1 | train 0.5577 | val 0.9414
epoch 2 | train 0.5109 | val 0.9622
epoch 3 | train 0.4993 | val 0.9976
epoch 4 | train 0.4932 | val 0.9170
epoch 5 | train 0.4852 | val 0.8292
epoch 6 | train 0.4843 | val 1.0669
epoch 7 | train 0.4820 | val 0.8828
epoch 8 | train 0.4797 | val 0.9023
best val loss: 0.8292
D00 val AUC: 0.516
D10 val AUC: 0.651
D20 val AUC: 0.646
D40 val AUC: 0.455


In [8]:
from sklearn.metrics import precision_recall_fscore_support

names = ["D00", "D10", "D20", "D40"]
grid = np.arange(0.05, 0.96, 0.05)

best_th = []
for i in range(4):
    f1s = [precision_recall_fscore_support(val_y[:, i], (val_probs[:, i] >= t).astype(int),
                                           average="binary", zero_division=0)[2] for t in grid]
    best_th.append(float(grid[int(np.argmax(f1s))]))
print("الحدود:", dict(zip(names, np.round(best_th, 2))))

pred = np.stack([(val_probs[:, i] >= best_th[i]).astype(int) for i in range(4)], axis=1)
p, r, f1, _ = precision_recall_fscore_support(val_y, pred, zero_division=0)
prev = val_y.mean(axis=0)
for i, n in enumerate(names):
    print(f"{n} | th {best_th[i]:.2f} | precision {p[i]:.2f} | recall {r[i]:.2f} | F1 {f1[i]:.2f} | مرجع {2*prev[i]/(1+prev[i]):.2f}")
print("macro-F1:", round(f1.mean(), 3))

الحدود: {'D00': np.float64(0.05), 'D10': np.float64(0.25), 'D20': np.float64(0.4), 'D40': np.float64(0.05)}
D00 | th 0.05 | precision 0.72 | recall 1.00 | F1 0.84 | مرجع 0.84
D10 | th 0.25 | precision 0.45 | recall 0.77 | F1 0.57 | مرجع 0.54
D20 | th 0.40 | precision 0.42 | recall 0.43 | F1 0.43 | مرجع 0.39
D40 | th 0.05 | precision 0.08 | recall 0.99 | F1 0.15 | مرجع 0.15
macro-F1: 0.496


In [9]:
import json
os.makedirs("/kaggle/working/artifacts_v2", exist_ok=True)
torch.save(model.state_dict(), "/kaggle/working/artifacts_v2/baseline_v2_resnet18.pt")
json.dump({"thresholds": dict(zip(names, best_th)), "seed": 42, "img_size": 384,
           "mean": mean, "std": std, "classes": names},
          open("/kaggle/working/artifacts_v2/baseline_v2_config.json", "w"), indent=2)
print(os.listdir("/kaggle/working/artifacts_v2"))

['baseline_v2_resnet18.pt', 'baseline_v2_config.json']


In [10]:
TEST = "/kaggle/input/datasets/sarvii7/roadscan-yolo/RoadScan_yolo/test_US"
print(len(os.listdir(f"{TEST}/images")), len(os.listdir(f"{TEST}/labels")))

4805 4805


In [11]:
def build_vectors_at(lbl_dir):
    stems, vecs = [], []
    for f in sorted(os.listdir(lbl_dir)):
        v = np.zeros(4, dtype=np.float32)
        for line in open(f"{lbl_dir}/{f}").read().strip().splitlines():
            if line.strip():
                v[int(line.split()[0])] = 1
        stems.append(f[:-4]); vecs.append(v)
    return stems, np.stack(vecs)

test_stems, test_y = build_vectors_at(f"{TEST}/labels")

class TestDS(RoadDataset):
    def __getitem__(self, i):
        img = Image.open(f"{TEST}/images/{self.stems[i]}.jpg").convert("RGB")
        return tf(img), torch.tensor(self.y[i])

test_dl = DataLoader(TestDS("test", test_stems, test_y), batch_size=64, shuffle=False, num_workers=2)
test_probs = predict(test_dl)

pred = np.stack([(test_probs[:, i] >= best_th[i]).astype(int) for i in range(4)], axis=1)
p, r, f1, _ = precision_recall_fscore_support(test_y, pred, zero_division=0)
prev = test_y.mean(axis=0)
for i, n in enumerate(names):
    print(f"{n} | AUC {roc_auc_score(test_y[:, i], test_probs[:, i]):.2f} | precision {p[i]:.2f} | recall {r[i]:.2f} | F1 {f1[i]:.2f} | مرجع {2*prev[i]/(1+prev[i]):.2f}")
print("macro-F1:", round(f1.mean(), 3))

D00 | AUC 0.57 | precision 0.81 | recall 1.00 | F1 0.90 | مرجع 0.90
D10 | AUC 0.56 | precision 0.55 | recall 0.14 | F1 0.22 | مرجع 0.66
D20 | AUC 0.61 | precision 0.25 | recall 0.33 | F1 0.28 | مرجع 0.27
D40 | AUC 0.61 | precision 0.02 | recall 0.99 | F1 0.05 | مرجع 0.05
macro-F1: 0.362
